In [1]:
import os
import cv2
import face_recognition
import numpy as np
import matplotlib.pyplot as plt


In [3]:
def load_and_encode_faces(data_dir):
    known_face_encodings = []
    known_face_names = []

    for person_name in os.listdir(data_dir):
        person_dir = os.path.join(data_dir, person_name)
        if os.path.isdir(person_dir):
            for img_name in os.listdir(person_dir):
                img_path = os.path.join(person_dir, img_name)
             
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                  image = face_recognition.load_image_file(img_path)
                  face_encoding = face_recognition.face_encodings(image)

                  if face_encoding:  # Ensure there are face encodings
                     known_face_encodings.append(face_encoding[0])
                     known_face_names.append(person_name)

    return known_face_encodings, known_face_names

data_directory = '/Users/macbook/Desktop/Photos/'  # Update this with your directory
known_face_encodings, known_face_names = load_and_encode_faces(data_directory)


In [5]:
def recognize_faces(image, threshold=0.6):
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    face_locations = face_recognition.face_locations(rgb_image)
    face_encodings = face_recognition.face_encodings(rgb_image, face_locations)

    face_names = []
    for face_encoding in face_encodings:
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=threshold)
        name = "Unknown"
        confidence = 0.0

        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        best_match_index = np.argmin(face_distances)

        if matches[best_match_index]:
            confidence = (1 - face_distances[best_match_index]) * 100
            
            # Only assign name if confidence is above 50%
            if confidence >= 50:
                name = known_face_names[best_match_index]

        face_names.append(f"{name} ({confidence:.2f}%)")

    return face_locations, face_names




In [7]:
video_capture = cv2.VideoCapture(0)  # Use 0 for webcam or a video file path

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    face_locations, face_names = recognize_faces(frame)

    # Draw bounding boxes and names
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        # Determine the color based on recognition
        if "Unknown" in name:
            color = (0, 0, 255)  # Red for unknown
        else:
            color = (0, 255, 0)  # Green for recognized

        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        cv2.putText(frame, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

    # Display the resulting frame
    cv2.imshow('Video', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()


